## Colab setup

Run this cell **first**. It clones the repository, installs dependencies,
copies the raw workbook across from Drive, and points the notebooks at the
clone.

**Put your GitHub token in Colab's secrets panel** (the key icon in the left
sidebar), named `GH_TOKEN`, with "Notebook access" switched on. Do not paste
it into a cell — a pasted token gets pushed to GitHub and GitHub will revoke
it automatically.

Safe to re-run. Outside Colab (local Jupyter) it does nothing, so the same
notebook works in both places.

**Colab wipes `/content` when the runtime disconnects.** Push before you
close the tab, or the run is lost — see the last cell of this notebook.


In [ ]:
# ============================================================
# COLAB SETUP — run first. No-op outside Colab. Safe to re-run.
# ============================================================
import os, sys, subprocess
from pathlib import Path

GH_USER    = "KT-Devv"
GH_REPO    = "student-dropout-prediction-ghana"
GH_BRANCH  = "main"
DRIVE_XLSX = "/content/drive/MyDrive/Ghana_Dropout_Project/ghana_dropout_study_M.xlsx"

GIT_NAME   = "Your Name"          # <-- edit
GIT_EMAIL  = "your@email"         # <-- edit

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

if not IN_COLAB:
    print("Not in Colab — skipping setup. Paths resolve from the repo root.")
else:
    def sh(cmd, check=True):
        r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
        if r.stdout.strip(): print(r.stdout.strip()[:2000])
        if check and r.returncode != 0:
            print(r.stderr.strip()[:2000])
        return r

    # ---- token from the secrets panel, never from a pasted string -------
    TOKEN = None
    try:
        from google.colab import userdata
        TOKEN = userdata.get("GH_TOKEN")
    except Exception:
        pass
    if not TOKEN:
        print("No GH_TOKEN secret found. Cloning read-only — you will be able "
              "to run, but NOT push.\n"
              "Add it: key icon in the left sidebar -> GH_TOKEN -> "
              "Notebook access on.")

    # ---- clone (or reuse an existing clone) -----------------------------
    REPO_PATH = Path(f"/content/{GH_REPO}")
    if REPO_PATH.exists():
        print(f"Repo already present at {REPO_PATH} — pulling latest.")
        sh(f"git -C {REPO_PATH} pull --ff-only", check=False)
    else:
        url = (f"https://{TOKEN}@github.com/{GH_USER}/{GH_REPO}.git" if TOKEN
               else f"https://github.com/{GH_USER}/{GH_REPO}.git")
        r = sh(f"git clone -b {GH_BRANCH} {url} {REPO_PATH}", check=False)
        if not REPO_PATH.exists():
            raise RuntimeError(
                "Clone failed. Check GH_USER/GH_REPO/GH_BRANCH above, and that "
                "your GH_TOKEN has Contents: Read and write on this repository."
            )

    os.chdir(REPO_PATH)
    os.environ["DROPOUT_REPO"] = str(REPO_PATH)
    if str(REPO_PATH) not in sys.path:
        sys.path.insert(0, str(REPO_PATH))

    # ---- dependencies ---------------------------------------------------
    if Path("requirements.txt").exists():
        print("installing requirements (quiet, ~1-2 min on a cold runtime)...")
        sh("pip install -q -r requirements.txt", check=False)

    # ---- raw data: the ONLY thing Drive is used for ---------------------
    # Pupil-level data is never committed (ethics: HuSSREC/AP/543/VOL. 5),
    # so it is copied in at runtime and .gitignore keeps it out of git.
    Path("data-raw").mkdir(exist_ok=True)
    target = Path("data-raw") / Path(DRIVE_XLSX).name
    if target.exists():
        print(f"raw workbook already present: {target}")
    else:
        try:
            from google.colab import drive
            if not os.path.exists("/content/drive/MyDrive"):
                drive.mount("/content/drive")
            if os.path.exists(DRIVE_XLSX):
                sh(f'cp "{DRIVE_XLSX}" data-raw/')
                print(f"copied raw workbook -> {target}")
            else:
                print(f"NOT FOUND: {DRIVE_XLSX}\n"
                      "Fix DRIVE_XLSX above, or upload the workbook to "
                      "data-raw/ manually. Notebooks 2-9 don't need it "
                      "(they read data-processed/cleaned_data.csv).")
        except Exception as e:
            print("Drive mount skipped:", e)

    # ---- git identity, needed before any commit -------------------------
    sh(f'git config user.name "{GIT_NAME}"', check=False)
    sh(f'git config user.email "{GIT_EMAIL}"', check=False)

    # ---- the check worth not skipping -----------------------------------
    r = subprocess.run("git status --porcelain", shell=True, text=True,
                       capture_output=True)
    leaked = [l for l in r.stdout.splitlines()
              if "data-raw" in l or "ghana_dropout_study" in l
              or "cleaned_data.csv" in l]
    if leaked:
        print("\n*** WARNING: pupil-level data is NOT being ignored by git ***")
        for l in leaked: print("   ", l)
        print("Do not commit until .gitignore covers these.")
    else:
        print("\ngit is correctly ignoring the raw data.")

    print(f"\nREPO : {os.getcwd()}")
    print(f"push : {'enabled' if TOKEN else 'DISABLED (no GH_TOKEN)'}")


# Notebook 1 — Data Cleaning

**Output:** `data-processed/cleaned_data.csv` plus a reconciled column
cascade and a missingness table.

## What changed from R01, and why

| Change | Reason |
|---|---|
| No `drive.mount()`, no hard-coded `PROJECT_DIR` | Q4 — no R01 notebook ran against a fresh clone |
| Identifier drop matches **whole tokens**, not substrings | `"id" in "residence_type"` is True. The R01 rule silently dropped any column containing those two letters |
| **No imputation, no encoding, no scaling here** | GATE-1(i). Every fitted transform now happens inside the fold (`pipeline.py`). This notebook only does operations that use fixed constants |
| Near-unique pruning restricted to text columns | The R01 rule dropped any column with >98% unique values, which targets IDs but catches continuous variables. A float attendance rate with 990 distinct values would have been deleted |
| Column cascade printed and committed | Q2, Q11 — nobody could reproduce 44 → 41 |
| Missingness and out-of-range counts computed | Q11 — M4 says these "were not formally recorded". They are recoverable |
| Leakage list consolidated into `config.LEAKAGE_EXACT` | R01 dropped leakage columns twice, in cells 14 and 22, with two different lists |

Cleaning is deliberately **conservative**: it removes only what cannot be a
predictor (identifiers, leakage, empty columns) and repairs only what is
unambiguously an error. Everything judgement-laden happens in the fold.

In [ ]:
# ---- bootstrap: repo-relative imports, no drive.mount, no hard-coded path ----
import sys, os
from pathlib import Path

def _find_repo(start=None):
    p = Path(start or Path.cwd()).resolve()
    for c in [p, *p.parents]:
        if (c / "config.py").exists():
            return c
    return p

REPO = Path(os.environ["DROPOUT_REPO"]) if os.environ.get("DROPOUT_REPO") else _find_repo()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

# In Colab, clone the repo first, then run:
#     import os; os.environ["DROPOUT_REPO"] = "/content/student-dropout-prediction-ghana"
# Raw pupil-level data is NOT in the repo (ethics); place it under data-raw/
# locally. Nothing below calls drive.mount().

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from config import *
from pipeline import binarise_target

banner("NOTEBOOK 1 — DATA CLEANING")
OUT = run_dir("notebook01_cleaning")
capture_environment(OUT)
print("outputs ->", OUT)

if RAW_WORKBOOK is None:
    raise FileNotFoundError(
        "Raw workbook not found. Expected data-raw/ghana_dropout_study_M.xlsx.\n"
        "Pupil-level data is not committed (ethics, M5), so place it there "
        "locally or set DROPOUT_REPO to a folder that contains it."
    )

df = pd.read_excel(RAW_WORKBOOK)
N_RAW_ROWS, N_RAW_COLS = df.shape
print(f"raw: {N_RAW_ROWS} rows x {N_RAW_COLS} columns   <-- Table 1 'Size (raw)'")

In [ ]:
# ---- 1. normalise column names ------------------------------------------
df.columns = (pd.Index(df.columns).astype(str)
              .str.strip().str.lower()
              .str.replace(r"\s+", "_", regex=True)
              .str.replace(r"[^a-z0-9_]", "", regex=True))
df = df.loc[:, ~df.columns.duplicated()]

# ---- 2. duplicate rows ---------------------------------------------------
n_dup = int(df.duplicated().sum())
print(f"duplicate rows: {n_dup}")
if n_dup:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"  removed -> {len(df)} rows")

assert TARGET in df.columns, f"target '{TARGET}' not found. Columns: {list(df.columns)}"

In [ ]:
# ---- 3. THE COLUMN CASCADE (Q2, Q11) ------------------------------------
# Precedence: leakage > identifier > empty, so the reason a reader needs is
# never hidden behind an accident of ordering.

cascade, n = [], df.shape[1]
cascade.append({"step": "0. raw workbook", "criterion": "-",
                "n_dropped": 0, "n_remaining": n, "columns": ""})

reasons = {c: drop_reason(c) for c in df.columns if c != TARGET}
leak    = [c for c, r in reasons.items() if r == "leakage"]
ident   = [c for c, r in reasons.items() if r and r.startswith("identifier")]
derived = [c for c, r in reasons.items() if r and r.startswith("derived duplicate")]
suspect = [c for c, r in reasons.items() if r and r.startswith("failed range")]
empty   = [c for c in df.columns
           if c != TARGET and c not in leak + ident + derived + suspect
           and df[c].isna().all()]

for step, crit, cols in [
    ("1. target leakage", "derived from or dated by the outcome", leak),
    ("2. identifier / provenance", "exact name or whole underscore token", ident),
    ("3. derived duplicate", "arithmetically recoverable from retained columns", derived),
    ("4. failed range validation", "values outside the declared scale", suspect),
    ("5. entirely empty", "100% missing", empty),
]:
    n -= len(cols)
    cascade.append({"step": step, "criterion": crit, "n_dropped": len(cols),
                    "n_remaining": n, "columns": "; ".join(cols)})

# text columns that are near-unique are identifiers the keyword rule missed.
# NOTE: restricted to text. The R01 rule applied to every column, so a
# continuous variable with many distinct values could have been deleted.
text_cols = [c for c in df.columns
             if c not in leak + ident + derived + suspect + empty and c != TARGET
             and is_text(df[c])]
near_unique = [c for c in text_cols if df[c].nunique(dropna=True) > 0.98 * len(df)]
n -= len(near_unique)
cascade.append({"step": "6. near-unique text", "criterion": ">98% distinct values, text only",
                "n_dropped": len(near_unique), "n_remaining": n,
                "columns": "; ".join(near_unique)})
cascade.append({"step": "7. constant / near-constant", "criterion": "pruned IN-FOLD on training statistics",
                "n_dropped": np.nan, "n_remaining": np.nan,
                "columns": "fold-dependent — see pipeline.preprocess_inside_fold"})

cascade_df = pd.DataFrame(cascade)
cascade_df.to_csv(OUT / "column_cascade.csv", index=False)
print(cascade_df[["step", "criterion", "n_dropped", "n_remaining"]].to_string(index=False))
print("\nreasons, column by column:")
for c in leak + ident + derived + suspect + near_unique:
    print(f"  {c:42s} <- {reasons.get(c, 'near-unique text')}")

# NOTE: derived duplicates and range-failed columns stay in cleaned_data.csv
# so a reader can inspect them; pipeline.preprocess_inside_fold() excludes
# them from every feature matrix via the same drop_reason() call.
df = df.drop(columns=[c for c in leak + ident + empty + near_unique
                      if c in df.columns], errors="ignore")
print(f"\nretained in cleaned_data.csv but EXCLUDED from every model: "
      f"{derived + suspect}")
print(f"\nafter cascade: {df.shape[1]} columns (incl. target)")

In [ ]:
# ---- 3b. what the OLD substring rule would have removed -----------------
OLD_KEYWORDS = ["id", "student_id", "study_id", "record_id", "serial",
                "index", "registration", "date_recorded", "enumerator_initials"]
raw_names = pd.read_excel(RAW_WORKBOOK, nrows=0).columns
raw_names = (pd.Index(raw_names).astype(str).str.strip().str.lower()
             .str.replace(r"\s+", "_", regex=True)
             .str.replace(r"[^a-z0-9_]", "", regex=True))

old_drop = {c for c in raw_names
            if any(k in c for k in OLD_KEYWORDS) and c != TARGET}
restored = sorted(old_drop - set(leak) - set(ident) - set(derived) - set(suspect))
if restored:
    print("*** RESTORED — the R01 substring rule dropped these silently: ***")
    for c in restored:
        print("   ", c)
    pd.DataFrame({"restored_column": restored}).to_csv(
        OUT / "columns_restored_vs_substring_rule.csv", index=False)
    print("\nThese are in the feature matrix now, so the matrix differs from "
          "every previously reported table. State it in M6.")
else:
    print("No column was lost to the R01 substring rule on this data.")

In [ ]:
# ---- 4. target to 0/1 ----------------------------------------------------
before = df[TARGET].value_counts(dropna=False)
df[TARGET] = binarise_target(df[TARGET])
unmapped = int(df[TARGET].isna().sum())
print("target, as recorded:\n", before.to_string())
if unmapped:
    raise ValueError(
        f"{unmapped} target values did not map. Add them to "
        f"pipeline.TARGET_LABEL_MAP rather than dropping the rows silently."
    )
df[TARGET] = df[TARGET].astype(int)
n_pos = int(df[TARGET].sum())
print(f"\ndropout {n_pos} / {len(df)} = {100*n_pos/len(df):.1f}%   "
      "<-- the base rate that must print beside every headline number")

In [ ]:
# ---- 5. MISSINGNESS (Q11) ------------------------------------------------
miss = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(df[c].dtype) for c in df.columns],
    "n_missing": [int(df[c].isna().sum()) for c in df.columns]})
miss["pct_missing"] = (100 * miss["n_missing"] / len(df)).round(2)
miss = miss.sort_values("n_missing", ascending=False)
miss.to_csv(OUT / "missingness_by_column.csv", index=False)

rows_any = int(df.isna().any(axis=1).sum())
print(f"records with >=1 missing value : {rows_any}/{len(df)} "
      f"({100*rows_any/len(df):.1f}%)   <-- M4 says this was never recorded")
print(f"columns with any missing       : {int((miss['n_missing']>0).sum())}")
print(miss[miss['n_missing'] > 0].to_string(index=False))

# The examiner's specific question: is the strongest feature imputed?
for c in ["average_exam_score", *ATTENDANCE_COLS]:
    if c in df.columns:
        print(f"  {c:26s} {100*df[c].isna().mean():5.1f}% missing "
              "(median-imputed in-fold)")

plt.figure(figsize=(10, 5))
top = miss[miss["n_missing"] > 0].head(20)
if len(top):
    sns.barplot(data=top, y="column", x="pct_missing", color="steelblue")
    plt.xlabel("% missing"); plt.title("Missingness by column")
    plt.tight_layout(); plt.savefig(OUT / "figures/missingness.png", dpi=200)
plt.close()

In [ ]:
# ---- 6. OUT-OF-RANGE VALUES (Q11) ---------------------------------------
# Table 2 shows term_1_attendance max 100.5 and term_2_attendance max 102.9.
# Attendance above 100% is a data error. R01 passed it to the model untreated.
rows = []
for c in ATTENDANCE_COLS:
    if c not in df.columns:
        continue
    s = pd.to_numeric(df[c], errors="coerce")
    rows.append({"column": c, "min": s.min(), "max": s.max(),
                 "n_above_100": int((s > ATTENDANCE_MAX).sum()),
                 "n_below_0": int((s < 0).sum()),
                 "n_exactly_0": int((s == 0).sum()),
                 "n_missing": int(s.isna().sum())})
out_df = pd.DataFrame(rows)
out_df.to_csv(OUT / "attendance_range_audit.csv", index=False)
print(out_df.to_string(index=False))
print(f"\nTOTAL out-of-range attendance values: "
      f"{int(out_df['n_above_100'].sum() + out_df['n_below_0'].sum())}")
print("\nTreatment: clipped to [0, 100] INSIDE the fold (pipeline.py step 4), "
      "using the physical bound rather than a data-derived threshold, so no "
      "information crosses the partition. Report the counts above in M6.")
print("\nNOTE: the R01 composite builder divided attendance by 100 only when "
      "the maximum exceeded 1.0, so values above 100 produced a NEGATIVE risk "
      "contribution and propagated into attendance_risk_index. Clipping first "
      "removes that.")

In [ ]:
# ---- 7. category audit ---------------------------------------------------
# Q3: engineered_data.csv contains both "High" and a misspelled "Hgh", which
# a label encoder treats as two distinct levels.
for col in CATEGORY_CANONICAL:
    if col in df.columns:
        vc = df[col].value_counts(dropna=False)
        print(f"\n{col} — as recorded:")
        print(vc.to_string())
        canon = set(CATEGORY_CANONICAL[col].values())
        seen = {str(v).strip() for v in vc.index if pd.notna(v)}
        unknown = {s for s in seen
                   if str(s).strip().lower() not in CATEGORY_CANONICAL[col]}
        if unknown:
            print(f"  *** NOT in CATEGORY_CANONICAL: {sorted(unknown)} ***")
            print("  Add them to config.CATEGORY_CANONICAL before proceeding — "
                  "an unmapped category becomes NaN and is imputed away.")
        print(f"  canonical target categories: {sorted(canon)}")
        print(f"  ordinal map: {ORDINAL_MAPS.get(col, 'NONE — label-encoded alphabetically!')}")

In [ ]:
# ---- 7b. CATEGORY AUDIT — run this before you trust any ordinal map ----
# Prints every categorical value and flags anything config.ORDINAL_MAPS
# misses. An unmapped category becomes NaN and is then imputed away, so a
# silent gap here loses part of a variable without telling anyone.
problems = audit_categories(df)
if len(problems):
    problems.to_csv(OUT / "category_audit_problems.csv", index=False)
    print("\n-> category_audit_problems.csv written. Fix config.py, re-run "
          "this notebook, and only then continue to Notebook 2.")
else:
    print("\nORDINAL_MAPS and NOMINAL_COLS cover every observed value.")

In [ ]:
# ---- 8. cluster audit (GATE-1 iv, Q2, Q18) ------------------------------
if SCHOOL_COL in df.columns:
    g = (df.groupby(SCHOOL_COL)
           .agg(n_pupils=(TARGET, "size"), n_dropout=(TARGET, "sum")))
    g["dropout_rate_pct"] = (100 * g["n_dropout"] / g["n_pupils"]).round(1)
    g["share_of_sample_pct"] = (100 * g["n_pupils"] / len(df)).round(1)
    g = g.sort_values("n_pupils", ascending=False)
    g.to_csv(OUT / "school_cluster_audit.csv")
    print(g.to_string())
    print(f"\neffective cluster count : {g.shape[0]} schools")
    print(f"largest school          : {g['share_of_sample_pct'].iloc[0]:.0f}% of the sample")
    print(f"dropout rate spread     : {g['dropout_rate_pct'].min():.1f}% "
          f"to {g['dropout_rate_pct'].max():.1f}%")
    print(f"\nconfig.SCHOOL_HANDLING = {SCHOOL_HANDLING!r}")
    print("M10/M18 must name this school count, not the metropolitan area.")
else:
    print(f"'{SCHOOL_COL}' not present — verify the column name in config.py")

In [ ]:
# ---- 9. save -------------------------------------------------------------
# cleaned_data.csv is intentionally UNIMPUTED and UNENCODED. Every fitted
# transform happens in pipeline.preprocess_inside_fold(). If you find
# yourself writing an "engineered_data.csv" for modelling, stop: that file
# is what failed GATE-1.
df.to_csv(CLEANED_CSV, index=False)
df.to_csv(OUT / "cleaned_data_snapshot.csv", index=False)

write_manifest(OUT, {
    "notebook": "01_cleaning",
    "raw_shape": [int(N_RAW_ROWS), int(N_RAW_COLS)],
    "cleaned_shape": [int(df.shape[0]), int(df.shape[1])],
    "n_positive": int(df[TARGET].sum()),
    "rows_with_missing": int(rows_any),
    "attendance_out_of_range": int(out_df["n_above_100"].sum() + out_df["n_below_0"].sum()) if len(out_df) else 0,
    "columns_restored_vs_substring_rule": restored,
    "dropped": {"leakage": leak, "identifier": ident,
                "empty": empty, "near_unique_text": near_unique},
})

print(f"saved  -> {CLEANED_CSV}")
print(f"shape  : {df.shape} (including target)")
print(f"\nNOT imputed, NOT encoded, NOT scaled — by design. "
      f"Predictors are assembled in-fold.")
print("\nNEXT: Notebook 2 (EDA on the training pool only).")

---

## Save your work

Colab wipes `/content` when the runtime disconnects. Run this before you
close the tab — including `results/`, which has to be committed (the previous
review failed partly because the diagnostic CSVs backing the reported tables
were not in the repository).


In [ ]:
# ---- commit and push this run ----
import os, subprocess, sys
if "google.colab" in sys.modules or os.path.exists("/content"):
    MESSAGE = "Notebook 1 Data Cleaning run"     # <-- edit if you like

    def sh(c):
        r = subprocess.run(c, shell=True, text=True, capture_output=True)
        print((r.stdout + r.stderr).strip()[:3000]); return r

    sh("git status --short")
    sh("git add -A")
    sh(f'git commit -m "{MESSAGE}"')
    r = sh("git push")
    if r.returncode != 0:
        print("\nPush failed. Usual causes: no GH_TOKEN secret, or the token "
              "lacks Contents: Read and write. Fix it and re-run this cell — "
              "the commit is already made locally, so nothing is lost until "
              "the runtime disconnects.")
else:
    print("Local run — commit with git as usual.")
